In [ ]:
# Example 1: Verify the primary public API surface is importable.
from dagster_pipes import (
    PipesContext,
    PipesContextData,
    PipesMessage,
    PipesParams,
    open_dagster_pipes,
)

for obj in (PipesContext, PipesContextData, PipesMessage, PipesParams, open_dagster_pipes):
    assert obj is not None
    print(f"{obj.__name__}: {type(obj).__name__}")


In [ ]:
# Example 2: Exercise the param loaders (env-var-based loader is the default).
# PipesMappingParamsLoader mirrors the env-var protocol: it reads the same
# DAGSTER_PIPES_CONTEXT / DAGSTER_PIPES_MESSAGES keys, holding encoded params.
from dagster_pipes import (
    PipesEnvVarParamsLoader,
    PipesCliArgsParamsLoader,
    PipesMappingParamsLoader,
    encode_env_var,
)

env_loader = PipesEnvVarParamsLoader()
cli_loader = PipesCliArgsParamsLoader()

# Without DAGSTER_PIPES_* env vars set, is_dagster_pipes_process() returns False.
assert env_loader.is_dagster_pipes_process() is False
assert cli_loader.is_dagster_pipes_process() is False
print(f"env_loader.is_dagster_pipes_process() = {env_loader.is_dagster_pipes_process()}")

mapping = {
    "DAGSTER_PIPES_CONTEXT": encode_env_var({"foo": "bar"}),
    "DAGSTER_PIPES_MESSAGES": encode_env_var({"path": "/tmp/messages.jsonl"}),
}
mapping_loader = PipesMappingParamsLoader(mapping)
assert mapping_loader.is_dagster_pipes_process() is True
assert mapping_loader.load_context_params() == {"foo": "bar"}
assert mapping_loader.load_messages_params() == {"path": "/tmp/messages.jsonl"}
print(f"mapping_loader context params: {mapping_loader.load_context_params()}")


In [ ]:
# Example 3: Full protocol round-trip — open a real PipesContext from encoded params
# (direct-data context + file message channel), report a materialization, then
# verify the messages the orchestrator side would consume.
import json, os, tempfile
from dagster_pipes import PipesMappingParamsLoader, encode_env_var, open_dagster_pipes

context_data = {
    "asset_keys": ["my_asset"], "code_version_by_asset_key": {"my_asset": None},
    "provenance_by_asset_key": {"my_asset": None}, "partition_key": None,
    "partition_key_range": None, "partition_time_window": None,
    "run_id": "run-abc", "job_name": "my_job", "retry_number": 0,
    "extras": {"my_key": "my_value"},
}
messages_path = os.path.join(tempfile.mkdtemp(), "messages.jsonl")
loader = PipesMappingParamsLoader({
    "DAGSTER_PIPES_CONTEXT": encode_env_var({"data": context_data}),
    "DAGSTER_PIPES_MESSAGES": encode_env_var({"path": messages_path}),
})

with open_dagster_pipes(params_loader=loader) as pipes:
    assert pipes.run_id == "run-abc" and pipes.job_name == "my_job"
    assert pipes.get_extra("my_key") == "my_value"
    pipes.report_asset_materialization(metadata={"rows": 42}, asset_key="my_asset")
    pipes.log.info("hello from dagster_pipes")

with open(messages_path, encoding="utf-8") as f:
    methods = [json.loads(line)["method"] for line in f.read().splitlines()]
print(f"captured methods: {methods}")
assert methods == ["opened", "report_asset_materialization", "log", "closed"], methods


In [ ]:
# Example 4: Boundary-crossing invariants — encode/decode round-trip (the zlib+base64
# param wire format) and both branches of PipesDefaultContextLoader (direct data vs file).
import json, os, tempfile
from dagster_pipes import (
    PipesDefaultContextLoader,
    decode_env_var, encode_env_var,
    decode_param, encode_param,
)

payload = {"run_id": "r1", "extras": {"n": 3, "nested": [1, 2, 3], "flag": True}}
assert decode_env_var(encode_env_var(payload)) == payload
assert decode_param(encode_param(payload)) == payload
print("encode/decode round-trips OK")

data = {"run_id": "run-file", "job_name": "file_job", "extras": {}}
ctx_path = os.path.join(tempfile.mkdtemp(), "context.json")
with open(ctx_path, "w", encoding="utf-8") as f:
    json.dump(data, f)

loader = PipesDefaultContextLoader()
with loader.load_context({"path": ctx_path}) as from_file:
    assert from_file["run_id"] == "run-file", from_file
with loader.load_context({"data": data}) as direct:
    assert direct == data
print("PipesDefaultContextLoader file + direct-data branches OK")


In [ ]:
# Example 5: Outside an orchestration process, open_dagster_pipes() warns and returns
# a no-op stand-in so business code runs unchanged. Probed in a fresh interpreter:
# in-kernel, an earlier cell's (closed) pipes singleton would otherwise be reused.
import os, subprocess, sys

probe = """
import warnings
from dagster_pipes import open_dagster_pipes
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with open_dagster_pipes() as pipes:
        pipes.log.info("running outside of Dagster")               # no-op, must not raise
        pipes.report_asset_materialization(metadata={"rows": 1})   # no-op, must not raise
assert any("not launched by a Dagster orchestration process" in str(w.message) for w in caught)
print("WARNED-AND-NOOPED")
"""
env = {k: v for k, v in os.environ.items() if not k.startswith("DAGSTER_PIPES_")}
result = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, env=env)
print(result.stdout.strip())
assert result.returncode == 0, result.stderr[-800:]
assert "WARNED-AND-NOOPED" in result.stdout
print("open_dagster_pipes() non-orchestrated round-trip OK")
